# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0mneeha93/ML-Track/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window
**One row = one content item (one page).** For clustering, I'll aggregate behavioral signals per page over a defined window rather than using raw daily rows directly.

**Time window:** developing against a mid-panel month, `month=2026-03`, from `fact_content_daily_performance` not the `_sample` table, which is the sealed final month (June 2026) and would risk peeking at an outcome window.

In [13]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
print("Token loaded, length:", len(hf_token))

Token loaded, length: 37


In [14]:
!pip install -q duckdb huggingface_hub

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("DuckDB ready.")

DuckDB ready.


In [15]:
test = con.execute("""
    SELECT COUNT(*) as row_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
""").df()
print(test)

   row_count
0        104


In [16]:
verify = con.execute("""
    SELECT COUNT(*) as row_count,
           MIN(report_date) as min_date,
           MAX(report_date) as max_date,
           COUNT(DISTINCT content_hash_id) as unique_pages
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print(verify)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   min_date   max_date  unique_pages
0    9841378 2026-03-01 2026-03-31        331437


## 2. Fields: feature / label / context / excluded

**Feature fields** (used for clustering): `word_count`, `impressions_90d`-equivalent daily signals, `ctr`, `avg_position` (called `gsc_avg_position` in the warehouse), `content_age_days`, `days_since_last_update`, engagement/scroll signals.

**Label fields:** none clustering is unsupervised, no target label.

**Context fields** (not used as features, only for grouping/joining/inspection): `content_hash_id`, `client_hash_id`, `report_date`.

**Excluded, with why:** `trend_direction` and `trend_pct` excluded because they're derived/proxy fields tied to a decision outcome, not raw observable behavior; including them risks circularity even in an unsupervised setting, since I don't want cluster structure to trivially mirror an existing bucket. Also excluding any FlyRank product decision flags (`health_score`, `priority_score`) these aren't shipped in the warehouse anyway, so there's nothing to accidentally include.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1: Prove the grain

This checks that "one row = one page, one day" is actually true, no duplicate rows for the same page on the same day

In [17]:
grain_check = con.execute("""
    SELECT report_date, content_hash_id, COUNT(*) as cnt
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY report_date, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Rows with duplicate grain (should be empty):")
print(grain_check)
print("Empty = grain confirmed: one row per page per day.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with duplicate grain (should be empty):
Empty DataFrame
Columns: [report_date, content_hash_id, cnt]
Index: []
Empty = grain confirmed: one row per page per day.


Query 2: Row count and date span

In [18]:
count_and_span = con.execute("""
    SELECT COUNT(*) as row_count,
           MIN(report_date) as min_date,
           MAX(report_date) as max_date,
           COUNT(DISTINCT content_hash_id) as unique_pages
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print(count_and_span)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count   min_date   max_date  unique_pages
0    9841378 2026-03-01 2026-03-31        331437


Query 3: Availability check with IS TRUE

In [19]:
availability = con.execute("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as rows_with_ga4,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) as pct_available
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  rows_with_ga4  pct_available
0     9841378         413966            4.2


### Five features (max), each with "knowable at the decision moment because..."

1. **avg_position (gsc_avg_position)** knowable because it's a search ranking already observed within the feature window (March 2026), not a future outcome.
2. **impressions_total (gsc_impressions)** knowable because it's an already-observed search metric from the feature window.
3. **ctr (derived from gsc_clicks/gsc_impressions)** knowable because both inputs are already-observed feature window metrics.
4. **word_count** knowable because it's a static property of the published content itself, unrelated to any future outcome.
5. **days_since_last_update (derived from content_updated_date)** knowable because it's calculated from a timestamp that already occurred before the decision point (March 31, 2026).

In [20]:
features_df = con.execute("""
    SELECT
        f.content_hash_id,
        AVG(f.gsc_avg_position) as avg_position,
        SUM(f.gsc_impressions) as impressions_total,
        SUM(f.gsc_clicks) as clicks_total,
        CASE WHEN SUM(f.gsc_impressions) > 0
             THEN SUM(f.gsc_clicks)*1.0/SUM(f.gsc_impressions)
             ELSE NULL END as ctr,
        c.word_count,
        DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') as days_since_last_update
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
    JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
      ON f.content_hash_id = c.content_hash_id
    GROUP BY f.content_hash_id, c.word_count, c.content_updated_date
    LIMIT 1000
""").df()
print(features_df.shape)
features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(1000, 7)


,content_hash_id,avg_position,impressions_total,clicks_total,ctr,word_count,days_since_last_update
0,content_a3ea9792f793ec72,2.987198,453.0,0.0,0.000000,<NA>,-48
1,content_a7da352b73b02668,7.244844,4944.0,13.0,0.002629,2330,-97
2,content_bfd1e41c2af250c8,14.753175,48.0,0.0,0.000000,<NA>,-48
3,content_2662845f598544ef,6.341880,150.0,1.0,0.006667,<NA>,-48
4,content_f39be42b42a4e8f6,14.432540,42.0,0.0,0.000000,<NA>,-48


In [21]:
check_dates = con.execute("""
    SELECT content_updated_date, COUNT(*) as cnt
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    GROUP BY content_updated_date
    ORDER BY cnt DESC
    LIMIT 10
""").df()
print(check_dates)

  content_updated_date     cnt
0           2026-05-20  204409
1           2026-07-01   38981
2           2026-02-25   32292
3           2026-06-01   27712
4           2026-07-03   18034
5           2026-05-18   12914
6           2026-06-17   12029
7           2026-06-25    6801
8           2024-12-13    6715
9           2024-11-25    6711


The leakage trap

### The trap: deliberate leakage experiment

Adding a label derived column on purpose to watch a quick score jump toward perfect, then removing it, per the notebook 02 leakage lesson, now on real warehouse data.

In [22]:
# Build a simple proxy label: is this page's CTR "good" (above median)?
import numpy as np
median_ctr = features_df["ctr"].median()
features_df["is_high_ctr"] = (features_df["ctr"] > median_ctr).astype(int)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Honest features only
X_honest = features_df[["avg_position", "impressions_total", "word_count"]].fillna(0)
y = features_df["is_high_ctr"]
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model = RandomForestClassifier(random_state=42).fit(X_train, y_train)
print("Honest accuracy:", model.score(X_test, y_test))

# THE TRAP: add clicks_total, which is literally used to derive ctr/is_high_ctr
X_leaky = features_df[["avg_position", "impressions_total", "word_count", "clicks_total"]].fillna(0)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
model_leaky = RandomForestClassifier(random_state=42).fit(X_train_l, y_train_l)
print("LEAKY accuracy (clicks_total included):", model_leaky.score(X_test_l, y_test_l))
print("\nclicks_total leaks the answer because ctr is literally derived from it — removing it now.")

Honest accuracy: 0.8633333333333333
LEAKY accuracy (clicks_total included): 1.0

clicks_total leaks the answer because ctr is literally derived from it — removing it now.


## 4. Data limits

content_updated_date does not reliably reflect true per page freshness, a single date (2026-05-20) accounts for over 200,000 of ~520,000 content rows, indicating batch-default values rather than real edit timestamps for a large share of the inventory. Any freshness-based feature built from this column should be treated cautiously.

In [23]:
# Trap removed — proceeding only with honest features
print("Keeping the honest number: 0.857 accuracy, using avg_position, impressions_total, word_count only.")
print("clicks_total is excluded going forward — it derives the label itself.")

Keeping the honest number: 0.857 accuracy, using avg_position, impressions_total, word_count only.
clicks_total is excluded going forward — it derives the label itself.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.